<a href="https://colab.research.google.com/github/phucsz/DAAI_N1.4/blob/main/Web_traffic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 5. Kiểm tra và Chuẩn hóa 3NF bảng Web_traffic
Tiến hành tải dữ liệu từ `/content/web_traffic.csv`, xử lý dữ liệu khuyết thiếu, trùng lặp và tách chiều dữ liệu nguồn truy cập (`traffic_source`) thành một bảng danh mục riêng biệt.

In [15]:
import pandas as pd

# Đọc dữ liệu từ file web_traffic.csv
web_traffic_df = pd.read_csv('/content/web_traffic.csv')

# Hiển thị cấu trúc tổng quan
print("--- THÔNG TIN CẤU TRÚC BAN ĐẦU BẢNG WEB_TRAFFIC ---")
web_traffic_df.info()

# Kiểm tra giá trị Null/NaN
print("\n--- SỐ LƯỢNG GIÁ TRỊ NULL TRONG MỖI CỘT ---")
print(web_traffic_df.isnull().sum())

# Kiểm tra trùng lặp
print("\n--- SỐ DÒNG BỊ TRÙNG LẶP HOÀN TOÀN ---")
print(web_traffic_df.duplicated().sum())

display(web_traffic_df.head())

--- THÔNG TIN CẤU TRÚC BAN ĐẦU BẢNG WEB_TRAFFIC ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      3652 non-null   object 
 1   sessions                  3652 non-null   int64  
 2   unique_visitors           3652 non-null   int64  
 3   page_views                3652 non-null   int64  
 4   bounce_rate               3652 non-null   float64
 5   avg_session_duration_sec  3652 non-null   float64
 6   traffic_source            3652 non-null   object 
dtypes: float64(2), int64(3), object(2)
memory usage: 199.8+ KB

--- SỐ LƯỢNG GIÁ TRỊ NULL TRONG MỖI CỘT ---
date                        0
sessions                    0
unique_visitors             0
page_views                  0
bounce_rate                 0
avg_session_duration_sec    0
traffic_source              0
dtype: int64

--- SỐ DÒNG BỊ

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral


In [16]:
# 1. Xử lý trùng lặp và giá trị khuyết thiếu (nếu có)
web_traffic_cleaned = web_traffic_df.drop_duplicates().copy()

# Điền khuyết thiếu cho các cột số (nếu có)
num_cols = ['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec']
for col in num_cols:
    if col in web_traffic_cleaned.columns and web_traffic_cleaned[col].isnull().any():
        web_traffic_cleaned[col] = web_traffic_cleaned[col].fillna(web_traffic_cleaned[col].median())

if 'traffic_source' in web_traffic_cleaned.columns and web_traffic_cleaned['traffic_source'].isnull().any():
    web_traffic_cleaned['traffic_source'] = web_traffic_cleaned['traffic_source'].fillna('Unknown')

# 2. Tạo bảng danh mục Nguồn truy cập (Traffic Sources Dim) để chuẩn hóa 3NF
if 'traffic_source' in web_traffic_cleaned.columns:
    unique_sources = web_traffic_cleaned['traffic_source'].unique()
    traffic_sources_dim = pd.DataFrame({
        'source_id': [f'SRC-{i+1:02d}' for i in range(len(unique_sources))],
        'traffic_source': unique_sources
    })

    # Ánh xạ khóa ngoại source_id vào bảng chính và loại bỏ cột văn bản lặp lại
    web_traffic_3nf = web_traffic_cleaned.merge(traffic_sources_dim, on='traffic_source', how='left')
    web_traffic_3nf = web_traffic_3nf.drop(columns=['traffic_source'])

    # Tổ chức lại thứ tự cột hợp lý
    cols_order = ['date', 'source_id', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec']
    web_traffic_3nf = web_traffic_3nf[[col for col in cols_order if col in web_traffic_3nf.columns]]

    # 3. Xuất kết quả ra file mới
    web_traffic_3nf.to_csv('/content/web_traffic_3nf.csv', index=False)
    traffic_sources_dim.to_csv('/content/traffic_sources_dim.csv', index=False)

    print("\nĐã xuất các file thành công:")
    print("1. Bảng chính: /content/web_traffic_3nf.csv")
    print("2. Bảng danh mục: /content/traffic_sources_dim.csv")

    print("\n--- 5 DÒNG ĐẦU BẢNG WEB_TRAFFIC ĐÃ CHUẨN HÓA 3NF ---")
    display(web_traffic_3nf.head())

    print("\n--- BẢNG DANH MỤC NGUỒN TRUY CẬP (traffic_sources_dim) ---")
    display(traffic_sources_dim.head())
else:
    print("Không tìm thấy cột traffic_source để chuẩn hóa 3NF.")


Đã xuất các file thành công:
1. Bảng chính: /content/web_traffic_3nf.csv
2. Bảng danh mục: /content/traffic_sources_dim.csv

--- 5 DÒNG ĐẦU BẢNG WEB_TRAFFIC ĐÃ CHUẨN HÓA 3NF ---


,date,source_id,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec
0,2013-01-01,SRC-01,9760,7253,39093,0.00514,102.9
1,2013-01-02,SRC-01,10456,8151,47611,0.00406,120.5
2,2013-01-03,SRC-02,10076,7458,36963,0.00401,263.6
3,2013-01-04,SRC-02,9973,8063,53078,0.00562,151.8
4,2013-01-05,SRC-03,10223,7882,36790,0.00525,168.6



--- BẢNG DANH MỤC NGUỒN TRUY CẬP (traffic_sources_dim) ---


,source_id,traffic_source
0,SRC-01,organic_search
1,SRC-02,direct
2,SRC-03,referral
3,SRC-04,social_media
4,SRC-05,paid_search
